Import Python modules

In [12]:
import pandas as pd
import numpy as np
import glob

Read in counts data

In [13]:
# Read in data
counts_df = pd.read_csv('../results/counts.csv', keep_default_na=False)
counts_df = counts_df.replace('', np.nan)

# Convert codon_site to string to avoid issues with rows with semicolon-separated codon sites
counts_df['codon_site'] = counts_df['codon_site'].astype(str)

counts_df.head()

/tmp/ipykernel_660844/1834990010.py:2: DtypeWarning: Columns (0: codon_position, 1: codon_site) have mixed types. Specify dtype option on import or set low_memory=False.
  counts_df = pd.read_csv('../results/counts.csv', keep_default_na=False)


,site,nt_mut,wt_nt,mut_nt,gene,codon_position,codon_site,wt_codon,mut_codon,wt_aa,...,syn_branch_length,mut_class,mut_type,subtype,segment,segment_subtype,segment_length,host,evo_opp,rate
0,71,A71C,A,C,PA,2,24,CAT,CCT,H,...,2339,nonsynonymous,AC,all,PA,PA_all,2148,all,1.088920,0.0
1,1022,A1022C,A,C,NA,2,341,AAT,ACT,N,...,403,nonsynonymous,AC,N2,NA,NA_N2,1407,human,0.286425,0.0
2,1022,A1022C,A,C,NA,2,341,AAC,ACC,N,...,53693,nonsynonymous,AC,N2,NA,NA_N2,1407,human,38.161336,0.0
3,166,A166C,A,C,NP,1,56,ATA,CTA,I,...,2,nonsynonymous,AC,all,NP,NP_all,1494,human,0.001339,0.0
4,166,A166C,A,C,NP,1,56,ATG,CTG,M,...,0,nonsynonymous,AC,all,NP,NP_all,1494,human,0.000000,NaN


Read in predicted rates from neutral model

In [14]:
predicted_rates_df = pd.read_csv(
    '../results/expected_rates.csv',
    keep_default_na=False
)
del predicted_rates_df['tau_squared']
predicted_rates_df.head()

,mut_type,segment,motif,predicted_rate
0,AC,HA,AAA,0.205293
1,AC,HA,AAC,0.283618
2,AC,HA,AAG,0.244312
3,AC,HA,AAT,0.317978
4,AC,HA,CAA,0.175258


Compute expected counts and subset the data to mutations with at least X actual or expected counts

In [15]:
counts_cutoff = 10
actual_expected_df = (
    counts_df
    .merge(predicted_rates_df, on=['mut_type', 'segment', 'motif'], how='left')
    .assign(expected_count=lambda x: x['predicted_rate'] * x['evo_opp'])
    # subset to rows with at least X actual or expected counts
    .query("actual_count >= @counts_cutoff | expected_count >= @counts_cutoff")
)
actual_expected_df.to_csv('../results/actual_expected.csv', index=False)
print("Number of unique nucleotide mutations with data:", len(actual_expected_df))
actual_expected_df.head()

Number of unique nucleotide mutations with data: 129762


,site,nt_mut,wt_nt,mut_nt,gene,codon_position,codon_site,wt_codon,mut_codon,wt_aa,...,mut_type,subtype,segment,segment_subtype,segment_length,host,evo_opp,rate,predicted_rate,expected_count
2,1022,A1022C,A,C,NA,2,341,AAC,ACC,N,...,AC,N2,NA,NA_N2,1407,human,38.161336,0.0,0.326890,12.474573
24,54,A54C,A,C,PA,3,18,GAA,GAC,E,...,AC,all,PA,PA_all,2148,all,83.464153,0.011981191537307355,0.210835,17.597170
50,33,A33C,A,C,HA,3,11,CTA,CTC,L,...,AC,H3,HA,HA_H3,1698,all,43.726737,0.22869302876845168,0.188505,8.242711
100,56,A56C,A,C,PA,2,19,AAG,ACG,K,...,AC,all,PA,PA_all,2148,all,48.735568,0.0,0.250907,12.228110
168,55,A55C,A,C,PA,1,19,AAG,CAG,K,...,AC,all,PA,PA_all,2148,all,48.510708,0.0,0.210835,10.227758


Compute fitness effects of single nucleotide mutations.

In [16]:
pseudo_count = 0.5
groupby_cols = [
    'host', 'subtype', 'segment', 'gene', 'site', 'wt_nt', 'mut_nt', 'nt_mut',
    'mut_class'
]
nt_fitness_df = (
    actual_expected_df
    .groupby(groupby_cols, as_index=False)
    .agg({'actual_count': 'sum', 'expected_count': 'sum'})
    .assign(
        delta_fitness=lambda x: \
            np.log((x['actual_count'] + pseudo_count) / (x['expected_count'] + pseudo_count))
    )

    .assign(
        delta_fitness=lambda x: \
            np.log((x['actual_count'] + pseudo_count) / (x['expected_count'] + pseudo_count))
    )
)
nt_fitness_df.to_csv('../results/nt_fitness_effects.csv', index=False)
print("Number of unique nt muts with estimated fitness effects:", len(nt_fitness_df))
nt_fitness_df.head()

Number of unique nt muts with estimated fitness effects: 102939


,host,subtype,segment,gene,site,wt_nt,mut_nt,nt_mut,mut_class,actual_count,expected_count,delta_fitness
0,all,H1,HA,HA,2,T,A,T2A,nonsynonymous,0,20.744908,-3.749264
1,all,H1,HA,HA,2,T,C,T2C,nonsynonymous,0,62.624230,-4.838252
2,all,H1,HA,HA,2,T,G,T2G,nonsynonymous,0,14.313739,-3.388702
3,all,H1,HA,HA,3,G,A,G3A,nonsynonymous,0,140.575438,-5.642442
4,all,H1,HA,HA,3,G,T,G3T,nonsynonymous,0,27.944332,-4.041096


Compute fitness effects of synonymous nucleotide mutations, aggregating data for all synonymous mutations at a given site.

In [17]:
groupby_cols = ['host', 'subtype', 'segment', 'gene', 'site']
site_syn_fitness_df = (
    actual_expected_df
    .query("mut_class == 'synonymous'")
    .groupby(groupby_cols, as_index=False)
    .agg({'actual_count': 'sum', 'expected_count': 'sum'})
    .assign(
        delta_fitness=lambda x: \
            np.log((x['actual_count'] + pseudo_count) / (x['expected_count'] + pseudo_count))
    )
)
site_syn_fitness_df.to_csv('../results/sitewise_synonymous_fitness_effects.csv', index=False)
print("Number of sites with estimated synonymous fitness effects:", len(site_syn_fitness_df))
site_syn_fitness_df.head()

Number of sites with estimated synonymous fitness effects: 17307


,host,subtype,segment,gene,site,actual_count,expected_count,delta_fitness
0,all,H1,HA,HA,6,139,177.735726,-0.245042
1,all,H1,HA,HA,9,42,64.868849,-0.430542
2,all,H1,HA,HA,13,87,133.083021,-0.423084
3,all,H1,HA,HA,15,51,119.129746,-0.842820
4,all,H1,HA,HA,16,10,5.199358,0.611022


Compute fitness effects of amino-acid mutations, aggregating data across nucleotide mutations that result in the same amino-acid mutation.

In [18]:
groupby_cols = [
    'host', 'subtype', 'segment', 'gene', 'codon_site', 'wt_aa', 'mut_aa', 'aa_mut',
    'mut_class'
]
aa_fitness_df = (
    actual_expected_df
    .groupby(groupby_cols, as_index=False)
    .agg({'actual_count': 'sum', 'expected_count': 'sum'})
    .assign(
        delta_fitness=lambda x: \
            np.log((x['actual_count'] + pseudo_count) / (x['expected_count'] + pseudo_count))
    )
)
aa_fitness_df.to_csv('../results/aa_fitness_effects.csv', index=False)
print("Number of unique aa muts with estimated fitness effects:", len(aa_fitness_df))
aa_fitness_df.head()

Number of unique aa muts with estimated fitness effects: 86380


,host,subtype,segment,gene,codon_site,wt_aa,mut_aa,aa_mut,mut_class,actual_count,expected_count,delta_fitness
0,all,H1,HA,HA,1,M,I,M1I,nonsynonymous,0,168.519770,-5.823163
1,all,H1,HA,HA,1,M,K,M1K,nonsynonymous,0,20.744908,-3.749264
2,all,H1,HA,HA,1,M,R,M1R,nonsynonymous,0,14.313739,-3.388702
3,all,H1,HA,HA,1,M,T,M1T,nonsynonymous,0,62.624230,-4.838252
4,all,H1,HA,HA,10,C,C,C10C,synonymous,11,8.955787,0.195720


Report the number of amino-acid level mutations with data in each category (including "synonymous" mutations that don't change the amino acid)

In [19]:
aa_fitness_df.groupby(['host', 'mut_class']).size()

host   mut_class    
all    nonsense          1917
       nonsynonymous    29497
       synonymous        6820
avian  nonsense           673
       nonsynonymous    12873
       synonymous        4757
human  nonsense          1592
       nonsynonymous    22852
       synonymous        5399
dtype: int64

In [20]:
aa_fitness_df[
    (aa_fitness_df['host'] == 'all') &
    (aa_fitness_df['mut_class'] == 'nonsynonymous')
].groupby(['segment', 'subtype']).size()

segment  subtype
HA       H1         2567
         H3         2464
         H5         1356
         H7           14
         H9          744
MP       all        1867
NA       N1         2167
         N2         2240
         N6          115
         N8          107
         N9           10
NP       all        2829
NS       all        1808
PA       all        3684
PB1      all        3572
PB2      all        3953
dtype: int64